# Lab 2: Four Data Traps

*Financial Engineering in Practice*, Chapter 2.

**What you will explore:** how the way returns are averaged, missing failures, look-ahead and tiny stocks change the return a backtest appears to earn.

You will measure four traps: the gap between arithmetic and geometric averages, survivorship bias, look-ahead bias, and the effect of micro-caps and equal weighting. Parts A, C and D use nearly a century of real U.S. data; Part B uses a simulated market in which the right answer is known, because free data does not include delisted stocks.

`python lab02.py` runs the same analysis in one go and writes a full report.

In [ ]:
from pathlib import Path
from IPython.display import Image, display
import lab02

SOURCE = "french"   # "french" downloads real data; "sim" uses fake data to test the code
OUT = Path("report")
OUT.mkdir(exist_ok=True)

data = lab02.load_all(SOURCE)
SIM = data["simulated"]
print("monthly:", data["monthly"].index[0], "to", data["monthly"].index[-1])
print("daily:  ", data["daily"].index[0].date(), "to", data["daily"].index[-1].date())
print("size deciles:", data["size_vw"].index[0], "to", data["size_vw"].index[-1], "(SIMULATED)" if SIM else "")

## Part A: Arithmetic versus geometric averages

**Predict first:** for the smallest 10% of stocks, how large will the gap between the arithmetic and geometric average be? Use the approximation σ²/2 from Section 2.1.

In [ ]:
a = lab02.part_a(data)
lab02.fig_averages(a, OUT, SIM)
display(Image(OUT / "fig2_1_averages.png"))
a.round(4)

## Part B: Survivorship, in a market where the truth is known

Every simulated stock has the same expected return. Stocks that fall far enough are delisted with a final return of −30%; others are acquired at a premium. We compare three datasets: the full, investable one; one missing the delisting returns; and one holding only the stocks that survived to the end.

**Predict first:** a strategy that buys the most volatile stocks has no true advantage here. What will the survivors-only dataset say about it?

In [ ]:
b = lab02.part_b(seed=2026)
print(b["stats"])
lab02.fig_survivorship(b, OUT, "")
display(Image(OUT / "fig2_2_survivorship.png"))
b["table"].round(4)

## Part C: Look-ahead bias

**C1.** A rule that holds stocks when the market closes above its 200-day average, otherwise Treasury bills. The correct version decides at yesterday's close; the look-ahead version decides at today's close but is credited with today's return.

In [ ]:
c = lab02.part_c(data["daily"], window=200)
lab02.fig_lookahead(c, OUT, SIM)
display(Image(OUT / "fig2_3_lookahead.png"))
c["performance"].round(3)

**C2.** Now choose the rule's window. With hindsight, pick the window with the best Sharpe ratio over the whole period. Walk-forward, pick each January using only the past.

In [ ]:
print(f"Hindsight: {c['hindsight_window']}-day window, Sharpe {c['hindsight_sharpe']:.2f}")
print(f"Walk-forward: Sharpe {c['walk_forward_sharpe']:.2f}")
lab02.fig_hindsight(c, OUT, SIM)
display(Image(OUT / "fig2_4_hindsight.png"))
c["chosen_windows"].tail(10)

## Part D: Micro-caps, weighting and the size premium

Banz published the size effect in 1981. How large is the premium in value- and equal-weighted portfolios, with and without the smallest decile, before and after publication?

In [ ]:
d = lab02.part_d(data["size_vw"], data["size_ew"], split="1980-12")
lab02.fig_size(d, OUT, SIM)
display(Image(OUT / "fig2_5_size_deciles.png"))
d["table"].round(3)

## Questions to think about

1. For each trap, write one sentence: how much did it change the result, and in which direction?
2. Does a typical free dataset represent what an investor could have done? What would you check first in someone else's backtest?
3. In Part B, run `lab02.part_b(seed=2026, delist_return=-0.55)`, closer to Nasdaq estimates. What changes?
4. In Part C, try `window=50` and `window=300`. Is the look-ahead gain larger for fast or slow rules? Why?